# Tanager Mangrove Mapping - 02 Classification

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** Pseudo-label generation from adaptive thresholds, RF + XGBoost training on Sangatta, accuracy evaluation, wall-to-wall extent map.

## 0. Environment Setup

In [ ]:
# Install dependencies (commented out for production)
# !pip install scikit-learn xgboost geopandas rasterio matplotlib joblib

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

# ============================================================
# Project root
# ============================================================
ROOT        = Path('..').resolve()
DATA_PROC   = ROOT / 'data' / 'processed'
DATA_GMW    = ROOT / 'data' / 'gmw_v3'
OUT_MODELS  = ROOT / 'outputs' / 'models'
OUT_RESULTS = ROOT / 'outputs' / 'results'
OUT_FIGURES = ROOT / 'outputs' / 'figures'

sys.path.insert(0, str(ROOT))
from src.preprocessing import load_geotiff_bands, compute_all_indices
from src.classification import (
    generate_pseudo_labels,
    build_feature_matrix,
    split_data,
    train_random_forest,
    train_xgboost,
    evaluate_model,
    compare_models,
    predict_extent,
    save_model
)

SCENE_ID = '20250302_030003_92_4001'   # Sangatta
print(f'ROOT         : {ROOT}')
print(f'Scene ID     : {SCENE_ID}')

## 1. Load Indices + Thresholds

In [ ]:
# ============================================================
# Load GeoTIFF bands and recompute indices
# ============================================================
data    = load_geotiff_bands(str(DATA_PROC), SCENE_ID)
indices = compute_all_indices(data)

# Load thresholds saved in 01_preprocessing.ipynb
thresh_path = OUT_RESULTS / f'thresholds_{SCENE_ID}.json'
with open(thresh_path) as f:
    thresholds = json.load(f)

print('Thresholds loaded:')
for k, v in thresholds.items():
    print(f'  {k:<8}: {v:.4f}')

## 2. Pseudo-label Generation

In [ ]:
# ============================================================
# AND logic: MVI > threshold AND NDMI > threshold
# ============================================================
labels = generate_pseudo_labels(indices, thresholds)

## 3. Feature Matrix + Train/Test Split

In [ ]:
X, y, feature_names = build_feature_matrix(indices, labels)
X_train, X_test, y_train, y_test = split_data(X, y)

## 4. Model Training

In [ ]:
# ============================================================
# Random Forest — primary model
# ============================================================
print('Training Random Forest...')
rf_model = train_random_forest(X_train, y_train)
save_model(rf_model, str(OUT_MODELS / f'rf_{SCENE_ID}.joblib'))

In [ ]:
# ============================================================
# XGBoost — comparison model
# ============================================================
print('Training XGBoost...')
xgb_model = train_xgboost(X_train, y_train)
save_model(xgb_model, str(OUT_MODELS / f'xgb_{SCENE_ID}.joblib'))

## 5. Evaluation

In [ ]:
rf_metrics  = evaluate_model(rf_model,  X_test, y_test, 'Random Forest')
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

In [ ]:
# ============================================================
# Comparison table — saved to outputs/results/
# ============================================================
comparison = compare_models(rf_metrics, xgb_metrics, y_test)
comparison.to_csv(OUT_RESULTS / f'accuracy_comparison_{SCENE_ID}.csv', index=False)
print(comparison)

## 6. Wall-to-Wall Extent Map

In [ ]:
# ============================================================
# Predict full scene using RF (primary model)
# ============================================================
h, w        = list(indices.values())[0].shape
extent_map  = predict_extent(rf_model, indices, original_shape=(h, w))

# Save as GeoTIFF
extent_path = DATA_PROC / f'extent_mangrove_{SCENE_ID}.tif'
with rasterio.open(
    extent_path, 'w',
    driver='GTiff', height=h, width=w,
    count=1, dtype='int8',
    crs=data['crs'], transform=data['transform'],
    compress='lzw'
) as dst:
    dst.write(extent_map, 1)

print(f'Extent map saved : {extent_path}')

## 7. Visualization

In [ ]:
# ============================================================
# Extent map + GMW v3 overlay
# ============================================================
gmw = gpd.read_file(DATA_GMW / 'gmw_sangatta.geojson')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# RF extent
axes[0].imshow(extent_map == 1, cmap='Greens')
axes[0].set_title('RF Mangrove Extent — Sangatta')
axes[0].axis('off')

# GMW v3 reference
gmw.plot(ax=axes[1], color='green', alpha=0.7)
axes[1].set_title('GMW v3 Reference — Sangatta')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(OUT_FIGURES / f'extent_map_{SCENE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()